## Environment Setup & Dependencies

In [7]:
import os
import cv2
import librosa
import numpy as np
import pandas as pd

from itertools import combinations

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout
)

## 1D Signal Processing & Feature Extraction
Define a robust extraction function to compute summary statistics (mean and standard deviation) for five core audio attributes:
* **MFCCs**: 16 bands resulting in 32 dimensions.
* **Spectral Centroid**: Brightness of the sound.
* **RMS**: Signal energy.
* **ZCR**: Zero-crossing rate.
* **Pitch (F0)**: Fundamental frequency extracted via the YIN algorithm (50Hz - 1000Hz).

In [8]:
def extract_features(y, sr):

    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=16)
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)

    mfcc_feature = np.concatenate([
        mfcc_mean,
        mfcc_std
    ])

    # shape = (32,)

    
    # SPECTRAL CENTROID
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    centroid_feature = np.array([
        np.mean(centroid),
        np.std(centroid)
    ])

    # shape = (2,)

    
    # RMS
    rms = librosa.feature.rms(y=y)
    rms_feature = np.array([
        np.mean(rms),
        np.std(rms)
    ])

    # shape = (2,)

    
    # ZCR
    zcr = librosa.feature.zero_crossing_rate(y)
    zcr_feature = np.array([
        np.mean(zcr),
        np.std(zcr)
    ])

    # shape = (2,)

    
    # PITCH
    f0 = librosa.yin(y, fmin=50, fmax=1000)
    f0 = f0[~np.isnan(f0)]

    if len(f0) > 0:
        pitch_feature = np.array([
            np.mean(f0),
            np.std(f0)
        ])

    else:
        pitch_feature = np.array([0, 0])

    # shape = (2,)

    return (
        mfcc_feature,
        centroid_feature,
        rms_feature,
        zcr_feature,
        pitch_feature
    )

## Data Ingestion & Preprocessing
Iterate through the directory structure to load raw audio files (`sr=16000`), extract the defined numerical features, and structure them into aligned lists mapping to their respective binary labels (Normal = 0, Sepsis = 1).

In [10]:
X_mfcc = []
X_centroid = []
X_rms = []
X_zcr = []
X_pitch = []

y_labels = []


def process_folder(folder_path, label):
    files = os.listdir(folder_path)
    for file in files:
        file_path = os.path.join(folder_path, file)

        try:
            # LOAD AUDIO
            y_audio, sr = librosa.load(
                file_path,
                sr=16000
            )

            # EXTRACT FEATURES
            (
                mfcc_feature,
                centroid_feature,
                rms_feature,
                zcr_feature,
                pitch_feature
            ) = extract_features(y_audio, sr)

            # SAVE FEATURES
            X_mfcc.append(mfcc_feature)
            X_centroid.append(centroid_feature)
            X_rms.append(rms_feature)
            X_zcr.append(zcr_feature)
            X_pitch.append(pitch_feature)
            y_labels.append(label)

        except Exception as e:
            print(f"Error processing {file_path}")
            print(e)


process_folder("dataset/normal", 0)
process_folder("dataset/sepsis", 1)

## Convert to NumPy Arrays

In [11]:
X_mfcc = np.array(X_mfcc)
X_centroid = np.array(X_centroid)
X_rms = np.array(X_rms)
X_zcr = np.array(X_zcr)
X_pitch = np.array(X_pitch)

y = np.array(y_labels)

print("MFCC Shape:", X_mfcc.shape)
print("Centroid Shape:", X_centroid.shape)
print("RMS Shape:", X_rms.shape)
print("ZCR Shape:", X_zcr.shape)
print("Pitch Shape:", X_pitch.shape)
print("Labels Shape:", y.shape)

MFCC Shape: (1817, 32)
Centroid Shape: (1817, 2)
RMS Shape: (1817, 2)
ZCR Shape: (1817, 2)
Pitch Shape: (1817, 2)
Labels Shape: (1817,)


## Combinatorial Feature Generation
Generate all $r$-length combinations of the extracted feature sets. This yields 31 unique configurations to comprehensively evaluate which combination of features holds the highest predictive power.

In [12]:
feature_dict = {
    "mfcc": X_mfcc,
    "centroid": X_centroid,
    "rms": X_rms,
    "zcr": X_zcr,
    "pitch": X_pitch
}

feature_names = list(feature_dict.keys())

all_combos = []

for r in range(1, len(feature_names) + 1):
    combos = combinations(feature_names, r)
    for combo in combos:
        all_combos.append(combo)

print("Total combinations:", len(all_combos))

print("\nAll combinations:\n")
for combo in all_combos:
    print(combo)

Total combinations: 31

All combinations:

('mfcc',)
('centroid',)
('rms',)
('zcr',)
('pitch',)
('mfcc', 'centroid')
('mfcc', 'rms')
('mfcc', 'zcr')
('mfcc', 'pitch')
('centroid', 'rms')
('centroid', 'zcr')
('centroid', 'pitch')
('rms', 'zcr')
('rms', 'pitch')
('zcr', 'pitch')
('mfcc', 'centroid', 'rms')
('mfcc', 'centroid', 'zcr')
('mfcc', 'centroid', 'pitch')
('mfcc', 'rms', 'zcr')
('mfcc', 'rms', 'pitch')
('mfcc', 'zcr', 'pitch')
('centroid', 'rms', 'zcr')
('centroid', 'rms', 'pitch')
('centroid', 'zcr', 'pitch')
('rms', 'zcr', 'pitch')
('mfcc', 'centroid', 'rms', 'zcr')
('mfcc', 'centroid', 'rms', 'pitch')
('mfcc', 'centroid', 'zcr', 'pitch')
('mfcc', 'rms', 'zcr', 'pitch')
('centroid', 'rms', 'zcr', 'pitch')
('mfcc', 'centroid', 'rms', 'zcr', 'pitch')


## MODEL FUNCTION

In [13]:
def build_numerical_model(input_dim):

    model = Sequential([

        Input(shape=(input_dim,)),

        Dense(128, activation='relu'),
        Dropout(0.3),

        Dense(64, activation='relu'),
        Dropout(0.2),

        Dense(32, activation='relu'),
        Dropout(0.1),

        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

## Ablation Experiment: Single-Seed Evaluation
Iterate through all 31 feature combinations to train and evaluate a Multilayer Perceptron (MLP). The dataset is standardized and split (80/20) using a single random seed (42). Predictions are derived using a strict 0.4 probability threshold.

In [14]:
results = []

for combo in all_combos:

    print("\n========================================")
    print("Training on:", combo)

    # CONCAT FEATURES
    X = np.concatenate(
        [feature_dict[f] for f in combo],
        axis=1
    )

    print("Combined Shape:", X.shape)

    # TRAIN TEST SPLIT
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # STANDARDIZATION
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # BUILD MODEL
    model = build_numerical_model(
        input_dim=X_train.shape[1]
    )

    # TRAIN MODEL
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_test, y_test),
        epochs=10,
        batch_size=32,
        verbose=0
    )

    # PREDICTIONS
    threshold = 0.4
    y_prob = model.predict(X_test, verbose=0)
    y_pred = (y_prob > threshold).astype(int).reshape(-1)

    # METRICS
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob.ravel())

    # SAVE RESULTS
    results.append({
        "Features": " + ".join(combo),
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": auc
    })

    # PRINT RESULTS
    print("Accuracy :", round(acc, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))
    print("ROC AUC  :", round(auc, 4))


Training on: ('mfcc',)
Combined Shape: (1817, 32)
Accuracy : 0.978
Precision: 0.9822
Recall   : 0.9708
F1 Score : 0.9765
ROC AUC  : 0.9986

Training on: ('centroid',)
Combined Shape: (1817, 2)
Accuracy : 0.6978
Precision: 0.6473
Recall   : 0.7836
F1 Score : 0.709
ROC AUC  : 0.7495

Training on: ('rms',)
Combined Shape: (1817, 2)
Accuracy : 0.7473
Precision: 0.6946
Recall   : 0.8246
F1 Score : 0.754
ROC AUC  : 0.8236

Training on: ('zcr',)
Combined Shape: (1817, 2)
Accuracy : 0.5412
Precision: 0.5076
Recall   : 0.7836
F1 Score : 0.6161
ROC AUC  : 0.6241

Training on: ('pitch',)
Combined Shape: (1817, 2)
Accuracy : 0.7198
Precision: 0.6683
Recall   : 0.8012
F1 Score : 0.7287
ROC AUC  : 0.7714

Training on: ('mfcc', 'centroid')
Combined Shape: (1817, 34)
Accuracy : 0.9835
Precision: 0.9882
Recall   : 0.9766
F1 Score : 0.9824
ROC AUC  : 0.9993

Training on: ('mfcc', 'rms')
Combined Shape: (1817, 34)
Accuracy : 0.9945
Precision: 0.9884
Recall   : 1.0
F1 Score : 0.9942
ROC AUC  : 0.9998

Tr

## CREATE RESULTS DATAFRAME

In [32]:
df = pd.DataFrame(results)
df = df.sort_values(
    by="Recall",
    ascending=False
)

# SHOW TOP RESULTS
print("\n========================================")
print("TOP 10 RESULTS")
print("========================================\n")

df.head(10)


TOP 10 RESULTS



,Features,Accuracy,Precision,Recall,F1,ROC_AUC
6,mfcc + rms,0.994505,0.988439,1.000000,0.994186,0.999818
27,mfcc + centroid + zcr + pitch,0.991758,0.988372,0.994152,0.991254,0.999303
17,mfcc + centroid + pitch,0.983516,0.976879,0.988304,0.982558,0.999242
15,mfcc + centroid + rms,0.983516,0.982456,0.982456,0.982456,0.999152
18,mfcc + rms + zcr,0.986264,0.988235,0.982456,0.985337,0.998788
8,mfcc + pitch,0.980769,0.976744,0.982456,0.979592,0.999212
25,mfcc + centroid + rms + zcr,0.989011,0.994083,0.982456,0.988235,0.999000
19,mfcc + rms + pitch,0.980769,0.982353,0.976608,0.979472,0.997182
20,mfcc + zcr + pitch,0.978022,0.976608,0.976608,0.976608,0.998818
30,mfcc + centroid + rms + zcr + pitch,0.978022,0.976608,0.976608,0.976608,0.998788


# SAVE RESULTS CSV

In [ ]:
df.to_csv(
    "numerical_feature_ablation_results.csv",
    index=False
)

print("\nResults saved successfully!")

## Robustness Testing: Multi-Seed Cross Validation
To ensure the feature combinations are stable and not overfitting to a specific train-test split, re-evaluate all 31 combinations across three distinct random states (42, 357, 999). Metrics are aggregated to report the mean performance.

In [47]:
random_states = [42, 357, 999]
results = []

for combo in all_combos:

    print("\n========================================")
    print("Training on:", combo)

    # CONCAT FEATURES
    X = np.concatenate(
        [feature_dict[f] for f in combo],
        axis=1
    )

    print("Combined Shape:", X.shape)

    # STORE METRICS OF 3 SPLITS
    acc_list = []
    precision_list = []
    recall_list = []
    f1_list = []
    auc_list = []

    
    # 3 DIFFERENT TRAIN-TEST SPLITS
    for rs in random_states:
        print(f"\n--- Random State: {rs} ---")

        # TRAIN TEST SPLIT
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=rs,
            stratify=y
        )

        # STANDARDIZATION
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # BUILD MODEL
        model = build_numerical_model(
            input_dim=X_train.shape[1]
        )

        # TRAIN MODEL
        model.fit(
            X_train,
            y_train,
            validation_data=(X_test, y_test),
            epochs=10,
            batch_size=32,
            verbose=0
        )

        # PREDICTIONS
        threshold = 0.4
        y_prob = model.predict(X_test, verbose=0)
        y_pred = (y_prob > threshold).astype(int).reshape(-1)

        # METRICS
        acc = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob.ravel())

        # SAVE CURRENT SPLIT METRICS
        acc_list.append(acc)
        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)
        auc_list.append(auc)

        # PRINT CURRENT SPLIT RESULTS
        print("Accuracy :", round(acc, 4))
        print("Precision:", round(precision, 4))
        print("Recall   :", round(recall, 4))
        print("F1 Score :", round(f1, 4))
        print("ROC AUC  :", round(auc, 4))

    
    # AVERAGE RESULTS OF 3 SPLITS
    avg_acc = np.mean(acc_list)
    avg_precision = np.mean(precision_list)
    avg_recall = np.mean(recall_list)
    avg_f1 = np.mean(f1_list)
    avg_auc = np.mean(auc_list)

    std_acc = np.std(acc_list)
    std_precision = np.std(precision_list)
    std_recall = np.std(recall_list)
    std_f1 = np.std(f1_list)
    std_auc = np.std(auc_list)

    # SAVE FINAL AVERAGE RESULT
    results.append({
        "Features": " + ".join(combo),
        "Accuracy": avg_acc,
        "Precision": avg_precision,
        "Recall": avg_recall,
        "F1": avg_f1,
        "ROC_AUC": avg_auc,
    })

    # PRINT FINAL AVG RESULTS
    print("\n===== AVERAGE RESULTS =====")

    print("Avg Accuracy :", round(avg_acc, 4))
    print("Avg Precision:", round(avg_precision, 4))
    print("Avg Recall   :", round(avg_recall, 4))
    print("Avg F1 Score :", round(avg_f1, 4))
    print("Avg ROC AUC  :", round(avg_auc, 4))


Training on: ('mfcc',)
Combined Shape: (1817, 32)

--- Random State: 42 ---
Accuracy : 0.978
Precision: 0.9822
Recall   : 0.9708
F1 Score : 0.9765
ROC AUC  : 0.9983

--- Random State: 357 ---
Accuracy : 0.9808
Precision: 0.9713
Recall   : 0.9883
F1 Score : 0.9797
ROC AUC  : 0.9981

--- Random State: 999 ---
Accuracy : 0.9643
Precision: 0.954
Recall   : 0.9708
F1 Score : 0.9623
ROC AUC  : 0.9917

===== AVERAGE RESULTS =====
Avg Accuracy : 0.9744
Avg Precision: 0.9692
Avg Recall   : 0.9766
Avg F1 Score : 0.9728
Avg ROC AUC  : 0.996

Training on: ('centroid',)
Combined Shape: (1817, 2)

--- Random State: 42 ---
Accuracy : 0.6951
Precision: 0.6456
Recall   : 0.7778
F1 Score : 0.7056
ROC AUC  : 0.7536

--- Random State: 357 ---
Accuracy : 0.7143
Precision: 0.6736
Recall   : 0.7602
F1 Score : 0.7143
ROC AUC  : 0.7364

--- Random State: 999 ---
Accuracy : 0.6429
Precision: 0.599
Recall   : 0.7251
F1 Score : 0.6561
ROC AUC  : 0.6779

===== AVERAGE RESULTS =====
Avg Accuracy : 0.6841
Avg Preci

## CREATE RESULTS_2 DATAFRAME

In [50]:
df2 = pd.DataFrame(results)
df2 = df2.sort_values(
    by="Recall",
    ascending=False
)

# SHOW TOP RESULTS
print("\n========================================")
print("TOP 10 RESULTS")
print("========================================\n")

df2.head(10)


TOP 10 RESULTS



,Features,Accuracy,Precision,Recall,F1,ROC_AUC
18,mfcc + rms + zcr,0.974359,0.965516,0.980507,0.972941,0.997051
19,mfcc + rms + pitch,0.976190,0.970986,0.978558,0.974755,0.997031
8,mfcc + pitch,0.976190,0.971079,0.978558,0.974748,0.995394
25,mfcc + centroid + rms + zcr,0.974359,0.967398,0.978558,0.972907,0.997041
17,mfcc + centroid + pitch,0.976190,0.972966,0.976608,0.974726,0.995788
0,mfcc,0.974359,0.969179,0.976608,0.972833,0.996031
7,mfcc + zcr,0.973443,0.969003,0.974659,0.971817,0.996425
28,mfcc + rms + zcr + pitch,0.970696,0.963424,0.974659,0.969003,0.997233
26,mfcc + centroid + rms + pitch,0.974359,0.970862,0.974659,0.972755,0.996485
15,mfcc + centroid + rms,0.972527,0.968964,0.972710,0.970787,0.995859


# SAVE RESULTS CSV

In [51]:
df2.to_csv(
    "numerical_feature_ablation_3x_results.csv",
    index=False
)

print("\nResults saved successfully!")


Results saved successfully!
